# SnapID (a versioned key-value store)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Hash Tables · **Difficulty/Frequency:** Uncommon (3/10)

> **Language note.** The official answer is Java; this notebook implements the same design in Python so every claim is executable. The Java reference is preserved verbatim in [`README.md`](README.md).

## Concepts

**What this problem is really testing:**
- **Store the deltas, not the states** — the single idea that turns O(n²) into O(n)
- **Tombstones**: representing a deletion as a *record*, not as an absence
- **Lower-bound binary search** over a history, to answer "what was true at time T?"

**First-principles primer — what is each piece?**

- **Versioned (or *temporal*) store.** An ordinary map answers "what is the value **now**?". This one answers "what was the value **at snapshot s**?" — so nothing can ever be overwritten or destroyed. Every write creates a new version and leaves the old ones readable.
- **snapID as logical time.** A monotonically increasing counter. It is not a clock — it is an *ordering*. That single property (strictly increasing) is what makes every per-key history automatically sorted, which is what makes binary search legal.
- **Tombstone.** A record that says *"deleted here"*. In an append-only history you cannot remove anything, so a deletion has to be **written down** like any other event. Reading a tombstone as the latest entry means "absent at this time" — which is different from "no history at all".
- **MVCC.** This is exactly **multi-version concurrency control**, the mechanism behind PostgreSQL, MongoDB's WiredTiger, and every database that lets a long-running read see a consistent snapshot while writers carry on. Naming it is worth real credit.

**The idea that makes it efficient:**

The obvious approach is to **copy the whole map** on every operation: O(n) per write, O(n²) total, and O(n²) memory. Hopeless.

But an operation only ever touches **one key**. So store just the change:

```
history[k1] = [(snap1, v1, alive), (snap3, -, TOMBSTONE), (snap4, v3, alive)]
```

Writes become O(1). Reads become "find the latest entry at or before snapID s" — and because snapIDs only increase, each list is **already sorted**, so that is a lower-bound binary search: **O(log m)**.

> **Store the deltas, not the states.** The same move behind git commits, event sourcing, and the [append-only log](../3.%20Persistent_Append_Only_Log/3.%20Persistent_Append_Only_Log.ipynb).

**Two absences that mean different things** — and this is the detail interviewers probe:

| Situation | `history[k]` | `hasKey(k, s)` |
|---|---|---|
| Key never inserted | missing entirely | `False` |
| Query *before* the key's first insert | exists, but no entry `<= s` | `False` |
| Latest entry `<= s` is a **tombstone** | exists, entry found | `False` |
| Latest entry `<= s` is a value | exists, entry found | `True` |

All three "no" cases must be handled, and only the last needs the value.

**Simple worked example.** The sequence from the prompt:

| op | snapID | `history[k1]` after |
|---|---|---|
| `insert(k1, v1)` | 1 | `[(1, v1)]` |
| `insert(k2, v2)` | 2 | unchanged — a *different* key |
| `delete(k1)` | 3 | `[(1, v1), (3, ✝)]` |
| `insert(k1, v3)` | 4 | `[(1, v1), (3, ✝), (4, v3)]` |

Now the queries. Each finds the **latest entry at or before** the given snapID:

- `hasKey(k1, 1)` → entry at 1 → a value → **True**
- `hasKey(k1, 2)` → *still* the entry at 1 (nothing happened to `k1` at 2) → **True**
- `hasKey(k1, 3)` → the tombstone → **False**
- `hasKey(k1, 4)` → the entry at 4 → **True**

Note `snap2` belongs to an operation on a *completely different key*, yet querying `k1` at `snap2` works — because snapIDs are **global** logical time, not per-key indices.

## Problem Statement

| Method | Behaviour |
|---|---|
| `insert(key, value) -> snapID` | Set the value; return the snapshot id this created |
| `delete(key) -> snapID` | Remove the key; return the snapshot id |
| `has_key(key, snapID)` | Did the key exist **at that snapshot**? |
| `get_val(key, snapID)` | Its value at that snapshot |

```python
ds = SnapStore()
s1 = ds.insert("k1", "v1")
s2 = ds.insert("k2", "v2")
s3 = ds.delete("k1")
s4 = ds.insert("k1", "v3")

ds.has_key("k1", s1)   # True
ds.has_key("k1", s2)   # True   - nothing happened to k1 at s2
ds.has_key("k1", s3)   # False  - the tombstone
ds.has_key("k1", s4)   # True
ds.get_val("k1", s1)   # "v1"
ds.get_val("k1", s4)   # "v3"
```

### Approach 1 — Naive (snapshot the whole map every time)

**Idea:** keep a list of complete copies of the map, one per operation. Lookup is then trivial: index into the list.

It is the most literal reading of "snapshot", and it is why the real solution exists. Copying the entire state on **every** write is O(n) per operation and O(n²) in both time and memory across n operations — a thousand keys and a thousand writes is a million stored entries.

**Time complexity:** **O(n) per write**, O(1) per read → **O(n²)** overall.

**Space complexity:** **O(n²)** — a full copy per operation.

In [ ]:
import bisect
from typing import Any, Dict, List, Optional, Tuple


class NaiveSnapStore:
    """Baseline: copies the entire map on every operation. O(n^2) time and space."""

    def __init__(self) -> None:
        self.snapshots: List[Dict[Any, Any]] = [{}]      # snapshots[0] is the empty state
        self.next_id = 1

    def insert(self, key: Any, value: Any) -> int:
        state = dict(self.snapshots[-1])                 # COPY EVERYTHING - the whole problem
        state[key] = value
        self.snapshots.append(state)
        self.next_id += 1
        return len(self.snapshots) - 1

    def delete(self, key: Any) -> int:
        state = dict(self.snapshots[-1])                 # ...again
        state.pop(key, None)
        self.snapshots.append(state)
        self.next_id += 1
        return len(self.snapshots) - 1

    def has_key(self, key: Any, snap_id: int) -> bool:
        return key in self.snapshots[snap_id]

    def get_val(self, key: Any, snap_id: int) -> Any:
        return self.snapshots[snap_id].get(key)

### Approach 2 — Optimal (per-key history + tombstones + binary search)

**Idea:** an operation touches one key, so record one entry: `(snapID, value, alive)`. Reading means finding the latest entry at or before the query snapID.

**Three details that carry it:**

- **The history is sorted for free.** `next_id` only ever increases, and entries are only ever *appended*, so each key's list is sorted by snapID by construction. That is what licenses binary search — and it is the step to say out loud, because "just binary search it" without justifying the sortedness leaves a hole in the argument.
- **`bisect_right((snap_id, HI)) - 1`** is the lower-bound idiom for "the last entry `<=` s". The sentinel second element makes an *exact* snapID match sort **after** the entry rather than before it, so a query at the exact moment of a write sees that write. Getting this backwards is a silent off-by-one that only shows up on exact-match queries.
- **A tombstone is a real record.** It occupies a snapID and sits in the history like any value. That is what makes delete-then-reinsert work with no special case: the timeline is just `value, tombstone, value`, and the binary search picks whichever is current.

**Deleting a key that was never inserted** still appends a tombstone. Harmless — a query lands on it and returns `False`, identical to having no history at all — and it keeps `delete` free of branches.

**Time complexity:** **O(1)** amortised per write; **O(log m)** per read, m = operations on *that* key.

**Space complexity:** **O(n)** total — exactly one entry per operation, across all keys.

In [ ]:
TOMBSTONE = object()          # a unique marker; no user value can ever equal it


class SnapStore:
    """Per-key append-only history + binary search. This is MVCC in miniature."""

    def __init__(self) -> None:
        # key -> list of (snap_id, value); value is TOMBSTONE for a deletion
        self.history: Dict[Any, List[Tuple[int, Any]]] = {}
        self.next_id = 1

    def _record(self, key: Any, value: Any) -> int:
        snap_id = self.next_id
        self.next_id += 1                          # strictly increasing => histories stay SORTED
        self.history.setdefault(key, []).append((snap_id, value))
        return snap_id

    def insert(self, key: Any, value: Any) -> int:
        return self._record(key, value)

    def delete(self, key: Any) -> int:
        return self._record(key, TOMBSTONE)        # a deletion is a RECORD, not an absence

    def _entry_at(self, key: Any, snap_id: int) -> Optional[Tuple[int, Any]]:
        """The latest entry with entry.snap_id <= snap_id, or None."""
        entries = self.history.get(key)
        if not entries:
            return None                            # the key was never touched at all
        # (snap_id, HIGH) sorts AFTER (snap_id, anything), so an exact match is included.
        idx = bisect.bisect_right(entries, (snap_id, _HIGH)) - 1
        return entries[idx] if idx >= 0 else None  # idx < 0 => the query predates the first write

    def has_key(self, key: Any, snap_id: int) -> bool:
        entry = self._entry_at(key, snap_id)
        return entry is not None and entry[1] is not TOMBSTONE

    def get_val(self, key: Any, snap_id: int, default: Any = None) -> Any:
        entry = self._entry_at(key, snap_id)
        if entry is None or entry[1] is TOMBSTONE:
            return default
        return entry[1]

    def latest_snap(self) -> int:
        return self.next_id - 1


class _High:
    """Sorts above every value, so bisect never compares two user values."""

    def __lt__(self, other): return False
    def __gt__(self, other): return True
    def __le__(self, other): return isinstance(other, _High)
    def __ge__(self, other): return True
    def __eq__(self, other): return isinstance(other, _High)
    def __hash__(self): return id(self)


_HIGH = _High()

### Follow-up — keys live at a snapshot, all versions of a key, and pruning

**Idea:** three natural extensions, each exercising a different property of the design.

- **`keys_at(snapID)`** — iterate every key and ask whether it was alive. O(K log m). Fine for a modest key count; if you need it often, keep a **global event log** of `(snapID, key)` so you can replay forward instead of probing every key.
- **`all_versions(key)`** — just read the history list. This is the payoff for keeping full history: it is already there, in order, and the only decision is whether to expose tombstones (usually yes — "when was it deleted?" is a real question).
- **`prune(before)`** — history grows forever, so real systems discard versions no reader can still see. The rule is subtle and worth stating precisely: for each key you must **keep the last entry at or before the cutoff**, because a query *at* the cutoff still needs it. Dropping everything strictly older would silently corrupt the boundary. PostgreSQL's `VACUUM` and MongoDB's oplog window solve exactly this.

**Time complexity:** `keys_at` O(K log m); `all_versions` O(m); `prune` O(n).

**Space complexity:** pruning is what bounds it.

In [ ]:
class SnapStorePlus(SnapStore):
    """Adds snapshot iteration, version history, and pruning."""

    def keys_at(self, snap_id: int) -> List[Any]:
        """Every key alive at that snapshot."""
        return sorted(k for k in self.history if self.has_key(k, snap_id))

    def all_versions(self, key: Any) -> List[Tuple[int, Any]]:
        """Every recorded version, oldest first. Tombstones shown as None."""
        return [(s, None if v is TOMBSTONE else v) for s, v in self.history.get(key, [])]

    def prune(self, before: int) -> int:
        """Discard versions no query at >= `before` could observe. Returns entries dropped."""
        dropped = 0
        for key, entries in list(self.history.items()):
            # KEEP the last entry at or before the cutoff - a query AT `before` still needs it.
            cut = bisect.bisect_right(entries, (before, _HIGH)) - 1
            if cut > 0:
                dropped += cut
                self.history[key] = entries[cut:]
            if self.history[key] and self.history[key][-1][1] is TOMBSTONE \
                    and len(self.history[key]) == 1 and self.history[key][0][0] <= before:
                # A lone tombstone older than the cutoff can go entirely.
                dropped += 1
                del self.history[key]
        return dropped

## Verification

The exact sequence from the problem statement, then the temporal cases that separate a correct implementation from a plausible one: queries between operations, at the exact moment of a write, before a key existed, and after delete-then-reinsert.

In [ ]:
import random

# --- The example from the problem statement, verbatim ---
ds = SnapStore()
s1 = ds.insert("k1", "v1")
s2 = ds.insert("k2", "v2")
s3 = ds.delete("k1")
s4 = ds.insert("k1", "v3")
assert (s1, s2, s3, s4) == (1, 2, 3, 4), "every operation gets its own snapID"

assert ds.has_key("k1", s1) is True
assert ds.has_key("k1", s2) is True, "s2 touched a DIFFERENT key; k1 is still alive"
assert ds.has_key("k1", s3) is False, "the tombstone"
assert ds.has_key("k1", s4) is True, "reinserted"
assert ds.get_val("k1", s1) == "v1"
assert ds.get_val("k1", s4) == "v3"

# The past is immutable: old snapshots still see old values
assert ds.get_val("k1", s2) == "v1", "s2 predates the delete, so v1 is still current there"
assert ds.get_val("k1", s3) is None, "deleted at s3"
assert ds.has_key("k2", s2) is True and ds.get_val("k2", s4) == "v2"

# --- Querying BEFORE a key existed ---
assert ds.has_key("k2", s1) is False, "k2 was not inserted until s2"
assert ds.get_val("k2", s1) is None
assert ds.has_key("never", s4) is False, "a key with no history at all"
assert ds.get_val("never", s4) is None

# --- Overwriting keeps every version readable ---
ov = SnapStore()
a = ov.insert("x", 1)
b = ov.insert("x", 2)
c = ov.insert("x", 3)
assert (ov.get_val("x", a), ov.get_val("x", b), ov.get_val("x", c)) == (1, 2, 3)
assert ov.has_key("x", a) and ov.has_key("x", c)

# --- Delete then reinsert, repeatedly: no special case needed ---
cycle = SnapStore()
snaps = []
for i in range(5):
    snaps.append(("ins", cycle.insert("k", i)))
    snaps.append(("del", cycle.delete("k")))
for kind, s in snaps:
    assert cycle.has_key("k", s) is (kind == "ins"), (kind, s)
for i, (kind, s) in enumerate(snaps):
    if kind == "ins":
        assert cycle.get_val("k", s) == i // 2

# --- Deleting a key that never existed is a harmless no-op ---
d = SnapStore()
s = d.delete("ghost")
assert d.has_key("ghost", s) is False
assert d.get_val("ghost", s) is None
after = d.insert("ghost", "now here")
assert d.has_key("ghost", after) is True
assert d.has_key("ghost", s) is False, "the earlier snapshot is unaffected"

# --- Falsy and None values are real values, distinct from absence ---
f = SnapStore()
for k, v in [("zero", 0), ("empty", ""), ("false", False), ("none", None)]:
    snap = f.insert(k, v)
    assert f.has_key(k, snap) is True, f"{k}: stored, therefore present"
    assert f.get_val(k, snap) == v or (v is None and f.get_val(k, snap) is None)
none_snap = f.insert("explicit_none", None)
assert f.has_key("explicit_none", none_snap) is True, (
    "a stored None is PRESENT - which is why has_key exists alongside get_val"
)
assert f.get_val("explicit_none", none_snap, default="MISSING") is None
assert f.get_val("absent", none_snap, default="MISSING") == "MISSING", (
    "the default distinguishes 'absent' from 'stored None'"
)

# --- Snapshot 0 predates everything ---
z = SnapStore()
s = z.insert("k", "v")
assert z.has_key("k", 0) is False, "snapID 0 is before any operation"
assert z.has_key("k", 999) is True, "a future snapID sees the latest state"

# --- Both implementations agree, on randomised operation sequences ---
random.seed(79)
for _ in range(300):
    fast, naive = SnapStore(), NaiveSnapStore()
    keys = [f"k{i}" for i in range(6)]
    snaps = []
    for _ in range(random.randint(1, 40)):
        k = random.choice(keys)
        if random.random() < 0.6:
            v = random.randint(0, 100)
            a, b = fast.insert(k, v), naive.insert(k, v)
        else:
            a, b = fast.delete(k), naive.delete(k)
        assert a == b, "the two must hand out identical snapIDs"
        snaps.append(a)

    for s in [0] + snaps:
        for k in keys:
            assert fast.has_key(k, s) == naive.has_key(k, s), (k, s)
            assert fast.get_val(k, s) == naive.get_val(k, s), (k, s)

# --- Space really is O(n), one entry per operation ---
sp = SnapStore()
for i in range(500):
    sp.insert(f"k{i % 10}", i)
assert sum(len(h) for h in sp.history.values()) == 500, "exactly one entry per operation"

# --- Follow-ups ---
plus = SnapStorePlus()
p1 = plus.insert("a", 1)
p2 = plus.insert("b", 2)
p3 = plus.delete("a")
p4 = plus.insert("c", 3)
assert plus.keys_at(p1) == ["a"]
assert plus.keys_at(p2) == ["a", "b"]
assert plus.keys_at(p3) == ["b"], "a was deleted at p3"
assert plus.keys_at(p4) == ["b", "c"]
assert plus.keys_at(0) == []

assert plus.all_versions("a") == [(p1, 1), (p3, None)], "the tombstone is visible in the history"
assert plus.all_versions("nothing") == []

# Pruning keeps the entry a boundary query still needs
pr = SnapStorePlus()
v1 = pr.insert("k", "old")
v2 = pr.insert("k", "mid")
v3 = pr.insert("k", "new")
assert pr.prune(before=v2) == 1, "only the entry strictly superseded before the cutoff goes"
assert pr.get_val("k", v2) == "mid", "the boundary query must still work after pruning"
assert pr.get_val("k", v3) == "new"
assert pr.get_val("k", v1) is None, (
    "a query BELOW the cutoff can no longer be answered - that is the price of pruning, "
    "and it is exactly MongoDB's 'resume token is no longer in the oplog'"
)
assert pr.has_key("k", v1) is False

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Iterating the keys alive at a snapshot.** `keys_at` above probes every key: O(K log m), fine when K is modest and hopeless when it is millions. The scalable version keeps a **global event log** of `(snapID, key, op)` alongside the per-key histories, so you replay forward from the last checkpoint instead of asking every key. That is the trade databases make: an extra write per operation to make snapshot iteration proportional to *changes*, not to the size of the keyspace.
- **Every version of a key.** Free — the history list *is* the answer. The only real decision is whether to include tombstones, and usually you should: "when was this deleted?" is exactly the question an audit log exists to answer.
- **Out-of-order queries.** Already handled: binary search does not care whether snapIDs arrive in order. The real question the follow-up is reaching for is **retention** — an old snapID stays valid only as long as you keep the versions it needs.
- **Pruning, and the boundary rule.** History grows without bound, so real systems discard versions no reader can still see. Two things follow, and both are worth stating.

  First, for each key you must **keep the last entry at or before the cutoff**, not merely everything strictly after it — a query *at* the cutoff still needs that entry. Dropping it corrupts the boundary silently.

  Second, and unavoidably: **snapIDs below the cutoff stop being answerable.** The assertions above show this directly — after `prune(before=v2)`, a query at `v1` returns "absent" rather than the old value, because the information is genuinely gone. That is not a bug; it is the price of bounded memory, and it is exactly MongoDB's *"resume token is no longer in the oplog"* and Kafka's `OFFSET_OUT_OF_RANGE`. A good implementation **detects and reports** the stale snapID rather than silently answering from whatever survived. PostgreSQL's `VACUUM` avoids the problem by tracking the oldest *live reader* and never pruning past it.
- **Compaction when a key churns.** If a key is written a thousand times and nobody ever queries the intermediate snapshots, those versions are garbage — but you cannot know that without tracking which snapIDs readers still hold. The practical answer is the same as pruning: maintain a **low-water mark** (the oldest snapID any live reader might use) and coalesce everything below it.
- **`LinkedList` vs `ArrayList`.** The follow-up is really about **random access**. Binary search needs O(1) indexing; a linked list forces an O(m) scan, so appends stay O(1) worst-case but reads degrade to linear. A dynamic array's append is O(1) *amortised* (occasional O(m) resize) and keeps reads at O(log m) — the right trade when reads dominate, which for a snapshot store they do.
- **This is MVCC.** Worth naming explicitly. PostgreSQL keeps multiple row versions tagged with transaction ids; MongoDB's WiredTiger does the same. It is what lets a long-running read see a stable snapshot while writers continue — readers never block writers and writers never block readers, because nothing is ever overwritten in place.

## Empirical complexity check

Compare **copying the whole map** (Approach 1) with **appending one delta** (Approach 2), over a growing number of operations spread across a fixed set of keys.

| Growth when the operation count doubles | What it means |
|---|---|
| ~4x | quadratic — each write copies a map that is itself growing |
| ~2x | linear — each write appends exactly one entry |

The memory difference is the same story: the naive store holds O(n²) entries, the delta store exactly n.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random


def make_ops(n):
    rng = random.Random(83)
    # Keys grow with n, so the naive copy cost grows too - the realistic shape.
    return ([(rng.random() < 0.8, f"k{rng.randrange(n // 4 + 1)}", i) for i in range(n)],)


def run_naive(ops):
    ds = NaiveSnapStore()
    for is_insert, k, v in ops:
        ds.insert(k, v) if is_insert else ds.delete(k)      # copies the ENTIRE map each time


def run_delta(ops):
    ds = SnapStore()
    for is_insert, k, v in ops:
        ds.insert(k, v) if is_insert else ds.delete(k)      # appends ONE entry


benchmark(
    {"Approach 1 - copy the whole map O(n^2)": run_naive,
     "Approach 2 - append one delta O(n)": run_delta},
    make_ops,
    sizes=[500, 1000, 2000, 4000],
    repeats=1,
)

# The memory difference, made concrete.
ops = make_ops(2000)[0]
naive, delta = NaiveSnapStore(), SnapStore()
for is_insert, k, v in ops:
    naive.insert(k, v) if is_insert else naive.delete(k)
    delta.insert(k, v) if is_insert else delta.delete(k)
naive_entries = sum(len(s) for s in naive.snapshots)
delta_entries = sum(len(h) for h in delta.history.values())
print(f"\nStored entries after {len(ops)} operations:")
print(f"  copy-everything : {naive_entries:>9,}")
print(f"  append-a-delta  : {delta_entries:>9,}   ({naive_entries / delta_entries:.0f}x smaller)")

## Patterns learned

- **Store the deltas, not the states.** Copying everything per change is O(n²); recording what changed is O(n). The same move powers git commits, event sourcing, the [append-only log](../3.%20Persistent_Append_Only_Log/3.%20Persistent_Append_Only_Log.ipynb), and every database's write-ahead log.
- **A deletion in an append-only world is a record, not an absence.** Tombstones let delete-then-reinsert work with zero special cases, and they preserve the answer to "when did it go away?".
- **A monotonically increasing counter gives you sortedness for free.** That is what makes binary search legal here — and saying *why* the list is sorted is the part of the correctness argument people skip.
- **"What was true at time T?" is a lower-bound search.** Find the latest event at or before T. The same shape as [Smallest Numbers](../20.%20Smallest_Numbers/20.%20Smallest_Numbers.ipynb) — and once you see a history as a sorted array, the technique transfers intact.
- **Distinguish the kinds of "not there".** Never inserted, not yet inserted at this time, and explicitly deleted are three different states. Add a stored `None` and you need `has_key` alongside `get_val` — the same sentinel problem as [Deep Key Search](../7.%20Deep_Key_Search_Nested_JSON/7.%20Deep_Key_Search_Nested_JSON.ipynb).
- **Unbounded history is a leak, and pruning costs you the past.** Keep the last version at or before the cutoff (a query *at* it still needs that entry), and accept that snapIDs below the cutoff become unanswerable. Detect and report that, rather than answering from whatever happened to survive.
- **Name the real-world pattern.** This is MVCC. Saying so connects a whiteboard exercise to how PostgreSQL and WiredTiger actually let readers and writers run without blocking each other.